### Tools 
Tools let agents take actions: things like fetching data, running code, calling external APIs, and even using a computer. The SDK supports five categories:

[Documentation](https://openai.github.io/openai-agents-python/tools/)

- **Hosted OpenAI tools**: execute for the model on OpenAI servers.
- **Local/runtime execution tools**: ComputerTool and ApplyPatchTool always run in your environment, while ShellTool can run locally or in a hosted container.
- **FunctionTool instances**: wrap any Python function as a tool.
- **Agents as tools**: expose an agent as a callable tool without a full handoff.
- **Experimental**: **Codex tool**: run workspace-scoped Codex tasks from a tool call.


In [1]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

from agents.decorators import tool

In [2]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [3]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [4]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [5]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=external_client
)

### Function Tools

In [6]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather of a given city."""
    return f"It's sunny in {city} today."

In [7]:
# Agent Creation
weather_agent: Agent = Agent(name="Weather Agent",
                     instructions="You are a helpful assistant.",
                     model=llm_model,
                     tools = [get_weather])

In [8]:
result = await Runner.run(weather_agent, "What is the Current weather in Lahore?")

print("\nCALLING AGENT\n")
print(result.final_output)


CALLING AGENT

It is currently sunny in Lahore.


In [19]:
from pprint import pprint
pprint(result)

RunResult(input='What is the Current weather in Lahore?',
          new_items=[ToolCallItem(agent=Agent(name='Weather Agent',
                                              handoff_description=None,
                                              tools=[FunctionTool(name='get_weather',
                                                                  description='Return '
                                                                              'the '
                                                                              'current '
                                                                              'weather '
                                                                              'of '
                                                                              'a '
                                                                              'given '
                                                                              'city.',
                        

### Hosted Tools
OpenAI offers a few built-in tools when using the OpenAIResponsesModel.

[List of Hosted Tools](https://openai.github.io/openai-agents-python/tools/#hosted-tools)

- **WebSearchTool**:
A hosted tool that lets the LLM search the web. Currently only supported with OpenAI models, using the Responses API. [See the details](https://openai.github.io/openai-agents-python/ref/tool/#agents.tool.WebSearchTool)

In [9]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [ ]:
from agents import WebSearchTool

agent = Agent(
    name="Assistant",
    model = 'gpt-4.1',
    tools=[WebSearchTool(search_content_types=["image", "text"],
                        image_settings={"max_results": 3, "caption": True})]
)


result = await Runner.run(agent, '''Find recent images and supporting text about the
current prices of petrol in Pakistan.''')

print(result.final_output)




Here is the most up‑to‑date information available regarding petrol (gasoline) prices in Pakistan as of today, Wednesday, October 7, 2026, along with a relevant image illustrating recent price displays:

Image (June 2026 display):  
A PSO (Pakistan State Oil) service station features a digital board indicating petrol at Rs 299.50 per litre and diesel at Rs 311.47 per litre, with a caption noting that these are "unofficial rates hold steady this week." This image reflects rates from **June 26, 2026** and serves as context for past pricing trends rather than current levels.citeturn0image0

 **Current Petrol Price (as of October 7, 2026)**  
- According to BaKhabar Pakistan's daily fuel‑price data, petrol in Pakistan is priced at **Rs 394.83 per litre** on October 7, 2026—a rise of Rs 1.19 from the previous day.([bakhabarpakistan.com](https://www.bakhabarpakistan.com/data/fuel-prices/?utm_source=openai))  
- This precise figure is corroborated by official reporting: Dawn’s Business sect

- **FileSearchTool**: 
The ```FileSearchTool``` allows retrieving information from your OpenAI Vector Stores.API.
[See the details](https://openai.github.io/openai-agents-python/ref/tool/#agents.tool.FileSearchTool)

In [14]:
from agents import FileSearchTool

agent = Agent(
    name="Assistant",
    model = 'gpt-4.1',
    tools=[FileSearchTool(max_num_results=3, vector_store_ids=["vs_6ac66e33d5148191b8c8b39e9d45706b"]), ]
)


result = await Runner.run(agent, '''who is Kamran and What was the last Qualification completed by him tell me about
the recent job experience?.''')

print(result.final_output)




Kamran is an AI and Data Science professional with experience as a Generative AI Instructor, AI Engineer, and Mentor, working with various reputable organizations and institutions such as the Punjab Judiciary Academy, Technitute, Gulzar Soft, and KICS (UET). He is involved in training judges, students, and professionals in AI, generative and agentic AI, as well as cloud and containerization technologies.

**Last Qualification Completed:**
- Kamran's most recently completed academic qualification is a "Masters in Information Technology" from the University of Education, Lahore, completed in August 2018. However, he is currently enrolled for a "Masters in Computer Science" at the University of Engineering and Technology (UET), Lahore, with an expected completion date of May 2026.

**Recent Job Experience:**
- Kamran's most recent ongoing job role is as a "Generative AI Instructor" at the Punjab Judiciary Academy (PJA), Lahore High Court, since June 2025. In this role, he has trained over